In [26]:
import pandas as pd
# from langchain_gigachat import GigaChat
from langgraph.graph import MessagesState
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langgraph.graph import START, StateGraph, END
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image, display
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from typing import Annotated, List, Tuple, Union, Literal, Dict
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_community.document_loaders.csv_loader import CSVLoader
import pandas as pd
import re
import subprocess
from tabulate import tabulate
import tqdm

In [27]:
load_dotenv()



True

In [28]:
data = pd.read_csv('scenarios_with_funcs.csv')
data

,scene_name,entry_condition,user_message,category,target,scene_text,funcs
0,Устранение проблем с блокировкой карты,Запрос пользователя связан с одной из перечисл...,"Я не могу использовать свою карту, она, похоже...",Проблемы с картами,1,Идентификация причины блокировки карты. Провер...,"[""check_transaction_history(), confirm_custome..."
1,Подтверждение транзакции,Запрос пользователя связан с одной из перечисл...,Я не вижу на своем счете транзакцию за 5000 ру...,Транзакции,1,Проверка данных транзакции. Верификация источн...,"[""verify_fund_source(), check_transaction_stat..."
2,Проверка статуса страхового полиса,Запрос пользователя связан с одной из перечисл...,Какой статус моего страхового полиса?,Страхование,1,Проверка идентификационной информации клиента....,"[""check_policy_status(), suggest_policy_terms(..."
3,Консультация по инвестиционным стратегиям,Запрос пользователя связан с одной из перечисл...,Какой у вас курс валюты на сегодня?,Инвестиции,0,Анализ финансовых целей клиента. Оценка рисков...,"[""suggest_investment_options()""]"
4,Оформление кредита,Запрос пользователя связан с одной из перечисл...,"Может, расскажете про кредиты? Интересно, что ...",Кредитование,1,Проверка кредитоспособности клиента. Сбор необ...,"[""collect_documents(), suggest_credit_products..."
...,...,...,...,...,...,...,...
190,Проверка активности счета,Запрос пользователя связан с одной из перечисл...,У меня на счету появилось какое-то подозритель...,Безопасность счета,1,Идентификация клиента. Запрос информации о пос...,"[""fetch_recent_transactions()""]"
191,Блокировка банковской карты при утере,Запрос пользователя связан с одной из перечисл...,"Слушай, кажется, я оставил карту в другом горо...",Проблемы с картами,1,Получение информации о состоянии карты. Провер...,"[""verify_client_data(), locate_last_activity()..."
192,Проверка статуса транзакции,Запрос пользователя связан с одной из перечисл...,"Я хочу узнать, прошла ли моя последняя транзак...",Транзакции,1,Получение информации о транзакции. Проверка ст...,"[""check_transaction_status()"", ""analyze_delay_..."
193,Проверка информации о страховом полисе,Запрос пользователя связан с одной из перечисл...,"У меня есть полис на квартиру, хочу узнать, ка...",Страхование,1,Проверка данных клиента. Получение информации ...,"[""fetch_policy_info(), verify_policy_status(),..."


In [29]:
import json
import pandas as pd

def decode_unicode_column(dataset, column_name='funcs'):
    """Декодирует значения в указанной колонке, если они в формате Unicode."""
    decoded_funcs = []
    for funcs in dataset[column_name]:
        if isinstance(funcs, str):
            try:
                decoded = json.loads(funcs)
                decoded_funcs.append(decoded)
            except json.JSONDecodeError:

                decoded_funcs.append(funcs)
        else:
            decoded_funcs.append(funcs)
    dataset[column_name] = decoded_funcs

decode_unicode_column(data, 'funcs')
data['funcs'].head()


0    [check_transaction_history(), confirm_customer...
1    [verify_fund_source(), check_transaction_statu...
2    [check_policy_status(), suggest_policy_terms()...
3                       [suggest_investment_options()]
4     [collect_documents(), suggest_credit_products()]
Name: funcs, dtype: object

In [30]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
import json

class Generator:
    def __init__(self, system_prompt_file=''):
        """
        Инициализация генератора с загрузкой системных промптов для разных режимов и API клиента.
        """
        load_dotenv()
        HF_API_KEY = os.getenv('HF_API_KEY')
        api_key = HF_API_KEY
        if api_key is None:
            raise ValueError("API key not found in environment variables.")

        self.client = OpenAI(
            base_url="https://api-inference.huggingface.co/models/Qwen/Qwen2.5-Coder-32B-Instruct/v1/",
            api_key=api_key,
        )

        if system_prompt_file:
            with open(system_prompt_file, "r", encoding="utf-8") as f:
                self.system_prompt = f.read()
        else:
            self.system_prompt = ""

    def prepare_message(self, scene_text, funcs):
        """
        Подготовка сообщения в нужном формате.
        """
        return json.dumps({
            "scen": scene_text,
            "funcs": funcs
        }, ensure_ascii=False)

    def generate(self, user_message, max_tokens=1500):
        """
        Генерирует ответ на основе пользовательского сообщения и системного промпта.
        """
        if not user_message:
            raise ValueError("User message cannot be empty.")

        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_message}
        ]

        try:
            completion = self.client.chat.completions.create(
                model="Qwen2.5-Coder-32B-Instruct",
                messages=messages,
                temperature=0.7,
                n=1,
                max_tokens=max_tokens,
            )

            return completion.choices[0].message.content

        except Exception as e:
            raise RuntimeError(f"Failed to generate response: {e}")

    def process_scenario(self, row):
        """
        Обработка одного сценария из датасета.

        :param row: Строка датафрейма с полями scene_text и funcs
        :return: Сгенерированный код функций
        """
        message = self.prepare_message(row['scene_text'], row['funcs'])
        return self.generate(message)
    
    
generator = Generator(system_prompt_file='system_prompt.txt')

In [31]:
def generate_tools():
    def get_random_ids(dataset, n=2):
        """Выбирает n случайных ID из датасета."""
        return dataset.sample(n).index.tolist()
    
    random_ids = get_random_ids(data, 2)
    generated_code_parts = [] 
    # print(random_ids)
    for idx in random_ids:
        try:
            row = data.loc[idx]
            generated_code = generator.process_scenario(row)
            generated_code_parts.append(generated_code) 
        except Exception as e:
            print(f"Error in row {idx}: {e}")
        
        full_code = "\n".join(generated_code_parts)
    

    return [full_code.replace(' @tool','@tool')], random_ids

for i in tqdm.tqdm(range(1,52)):
    results, idxs = generate_tools()
    save_path = "generated_tools.py"
    with open(save_path, "a") as file:
        for line in results:
            cleaned_line = line.replace("\\n", "\n").replace("\\t", "\t")
            file.write(cleaned_line + "\n")

100%|██████████| 51/51 [14:25<00:00, 16.97s/it]
